<h1>Objective</h1><ul><li> How to use linear classifier in pytorch.</li></ul> 


<h1>Linear  Classifier with PyTorch </h1>


<p>Before you use a  Deep neural network to solve the classification problem,  it 's a good idea to try and solve the problem with the simplest method. You will need the dataset object from the previous section.
In this lab, we solve the problem with a linear classifier.
 You will be asked to determine the maximum accuracy your linear classifier can achieve on the validation data for 5 epochs. We will give some free parameter values if you follow the instructions you will be able to answer the quiz. Just like the other labs there are several steps, but in this lab you will only be quizzed on the final result. </p>


<h2>Table of Contents</h2>


<div class="alert alert-block alert-info" style="margin-top: 20px">


<ul>
    <li><a href="#auxiliary"> Imports and Auxiliary Functions </a></li>
    <li><a href="#download_data"> Download data</a></li>
    <li><a href="#data_class"> Dataset Class</a></li>
    <li><a href="#trasform_Data_object">Transform Object and Dataset Object</a></li>
    <li><a href="#Question">Question</a></li>
</ul>
<p>Estimated Time Needed: <strong>25 min</strong></p>
 </div>
<hr>


<h2 id="auxiliary">Imports and Auxiliary Functions</h2>


The following are the libraries we are going to use for this lab:


In [1]:
!pip install skillsnetwork
from PIL import Image
import matplotlib.pyplot as plt
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn as nn
from torch import optim 
import skillsnetwork 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 8.1.5
    Uninstalling ipywidgets-8.1.5:
      Successfully uninstalled ipywidgets-8.1.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


<h2 id="download_data">Download Data</h2>


In this section, you are going to download the data from IBM object storage using **skillsnetwork.prepare** command. <b>skillsnetwork.prepare</b> is a command that's used to download a zip file, unzip it and store it in a specified directory. Locally we store the data in the directory  **/resources/data**. 


First, we download the file that contains the images:


In [3]:
await skillsnetwork.prepare("https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0321EN/data/images/concrete_crack_images_for_classification.zip", path = "./data", overwrite=True)

  0%|          | 0/40000 [00:00<?, ?it/s]

Saved to 'data'


<h2 id="data_class">Dataset Class</h2>


In this section, we will use the previous code to build a dataset class. As before, make sure the even samples are positive, and the odd samples are negative.  In this case, if the parameter <code>train</code> is set to <code>True</code>, use the first 10 000 samples as training data; otherwise, the last 10 000 samples will be used as validation data. Do not forget to sort your files so they are in the same order.  


**Note:** We are using the first 10,000 samples as our training data instead of the available 30,000 to decrease the training time of the model. If you want, you can train it yourself with all 30,000 samples just by modifying 2 lines in the following code chunk.


In [7]:
class Dataset(Dataset):

    # Constructor
    def __init__(self,transform=None,train=True):
        directory="./data"
        positive="Positive"
        negative="Negative"

        positive_file_path=os.path.join(directory,positive)
        negative_file_path=os.path.join(directory,negative)
        positive_files=[os.path.join(positive_file_path,file) for file in  os.listdir(positive_file_path) if file.endswith(".jpg")]
        positive_files.sort()
        negative_files=[os.path.join(negative_file_path,file) for file in  os.listdir(negative_file_path) if file.endswith(".jpg")]
        negative_files.sort()
        number_of_samples=len(positive_files)+len(negative_files)
        self.all_files=[None]*number_of_samples
        self.all_files[::2]=positive_files
        self.all_files[1::2]=negative_files 
        # The transform is goint to be used on image
        self.transform = transform
        #torch.LongTensor
        self.Y=torch.zeros([number_of_samples]).type(torch.LongTensor)
        self.Y[::2]=1
        self.Y[1::2]=0
        
        if train:
            self.all_files=self.all_files[0:10000] #Change to 30000 to use the full test dataset
            self.Y=self.Y[0:10000] #Change to 30000 to use the full test dataset
            self.len=len(self.all_files)
        else:
            self.all_files=self.all_files[30000:]
            self.Y=self.Y[30000:]
            self.len=len(self.all_files)    
       
    # Get the length
    def __len__(self):
        return self.len
    
    # Getter
    def __getitem__(self, idx):
        
        
        image=Image.open(self.all_files[idx])
        y=self.Y[idx]
          
        
        # If there is any transform method, apply it onto the image
        if self.transform:
            image = self.transform(image)

        return image, y

<h2 id="trasform_Data_object">Transform Object and Dataset Object</h2>


Create a transform object, that uses the <code>Compose</code> function. First use the transform <code>ToTensor()</code> and followed by <code>Normalize(mean, std)</code>. The value for <code> mean</code> and <code>std</code> are provided for you.


In [8]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
# transforms.ToTensor()
#transforms.Normalize(mean, std)
#transforms.Compose([])

transform =transforms.Compose([ transforms.ToTensor(), transforms.Normalize(mean, std)])


Create object for the training data  <code>dataset_train</code> and validation <code>dataset_val</code>. Use the transform object to convert the images to tensors using the transform object:


In [9]:
dataset_train=Dataset(transform=transform,train=True)
dataset_val=Dataset(transform=transform,train=False)

We  can find the shape of the image:


In [10]:
dataset_train[0][0].shape

torch.Size([3, 227, 227])

We see that it's a color image with three channels:


In [11]:
size_of_image=3*227*227
size_of_image

154587

<h2 id="Question"> Question <h2>


<b> Create a custom module for Softmax for two classes,called model. The input size should be the <code>size_of_image</code>, you should record the maximum accuracy achieved on the validation data for the different epochs. For example if the 5 epochs the accuracy was 0.5, 0.2, 0.64,0.77, 0.66 you would select 0.77.</b>


Train the model with the following free parameter values:


<b>Parameter Values</b>
   <li>learning rate:0.1 </li>
   <li>momentum term:0.1 </li>
   <li>batch size training:5</li>
   <li>Loss function:Cross Entropy Loss </li>
   <li>epochs:5</li>
   <li>set: torch.manual_seed(0)</li>


In [12]:
torch.manual_seed(0)

<b>Custom Module:</b>


In [16]:
import torch
import torch.nn as nn

class SoftmaxClassifier(nn.Module):
    def __init__(self, input_size, num_classes=2):
        super(SoftmaxClassifier, self).__init__()
        # Solo una capa lineal que toma el vector de píxeles y devuelve logits para 2 clases
        self.fc = nn.Linear(input_size, num_classes)

    def forward(self, x):
        # x vendrá con forma [batch_size, 3, 227, 227]; lo aplanamos a [batch_size, input_size]
        x = x.view(x.size(0), -1)
        logits = self.fc(x)
        return logits


<b>Model Object:</b>


In [17]:
# Suponiendo que ya definiste:
# size_of_image = 3 * 227 * 227
# (eso aparece justo antes de la sección “Question” en el notebook)

model = SoftmaxClassifier(input_size=size_of_image, num_classes=2)


<b>Optimizer:</b>


In [20]:
import torch.optim as optim

optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.1)


<b>Criterion:</b>


In [21]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()


<b>Data Loader Training and Validation:</b>


In [23]:
from torch.utils.data import DataLoader

# Suponiendo que ya definiste dataset_train y dataset_val justo antes:
# dataset_train = Dataset(transform=transform, train=True)
# dataset_val   = Dataset(transform=transform, train=False)

train_loader = DataLoader(dataset=dataset_train, batch_size=5, shuffle=True)
val_loader   = DataLoader(dataset=dataset_val,   batch_size=5, shuffle=False)


<b>Train Model with 5 epochs, should take 35 minutes: </b>


In [24]:
import torch

# Decidimos entrenar en CPU o GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward
        outputs = model(images)                # logits de tamaño [batch_size, 2]
        loss = criterion(outputs, labels)

        # Backward + optimizar
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Estadísticas de entrenamiento
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += torch.sum(preds == labels.data).item()
        total_samples += images.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc  = running_corrects / total_samples

    # Validación en cada época
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    val_samples = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)

            val_outputs = model(val_images)
            v_loss = criterion(val_outputs, val_labels)

            val_loss += v_loss.item() * val_images.size(0)
            _, val_preds = torch.max(val_outputs, 1)
            val_corrects += torch.sum(val_preds == val_labels.data).item()
            val_samples += val_images.size(0)

    val_epoch_loss = val_loss / val_samples
    val_epoch_acc  = val_corrects / val_samples

    print(
        f"Epoch [{epoch+1}/{num_epochs}]  "
        f"Train Loss: {epoch_loss:.4f}  Train Acc: {epoch_acc:.4f}  "
        f"Val Loss: {val_epoch_loss:.4f}  Val Acc: {val_epoch_acc:.4f}"
    )


Epoch [1/5]  Train Loss: 1146.9301  Train Acc: 0.8110  Val Loss: 1269.6031  Val Acc: 0.8016
Epoch [2/5]  Train Loss: 752.6127  Train Acc: 0.8611  Val Loss: 1012.9014  Val Acc: 0.8315
Epoch [3/5]  Train Loss: 711.9025  Train Acc: 0.8681  Val Loss: 1599.7069  Val Acc: 0.8027
Epoch [4/5]  Train Loss: 655.2340  Train Acc: 0.8751  Val Loss: 857.3785  Val Acc: 0.8369
Epoch [5/5]  Train Loss: 656.0736  Train Acc: 0.8742  Val Loss: 2934.2944  Val Acc: 0.7360


<h2>About the Authors:</h2>
 <a href=\"https://www.linkedin.com/in/joseph-s-50398b136/\">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.



## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-18  | 2.0  | Shubham  |  Migrated Lab to Markdown and added to course repo in GitLab |



Copyright &copy; 2019 <a href="cognitiveclass.ai"> cognitiveclass.ai</a>. This notebook and its source code are released under the terms of the <a href="https://bigdatauniversity.com/mit-license/">MIT License</a>
